In [0]:
snowflake_options = {
    "sfUrl": "gj27919.south-central-us.azure.snowflakecomputing.com",
    "sfUser": "Food_USER",
    "sfPassword": "snowflake123#",
    "sfDatabase": "Food_DB",
    "sfSchema": "Food_SCHEMA",
    "sfWarehouse": "Food_WH"
}


In [0]:
# Read Dallas staging table
df_dallas = spark.read \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "STG_DALLAS") \
    .load()

# Read Chicago staging table
df_chicago = spark.read \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "STG_CHICAGO") \
    .load()


In [0]:
df_dallas_sel = df_dallas.selectExpr(
    "STREET_ADDRESS as Address", "CITY", "STREET_DIRECTION",
    "ZIP_CODE as Zip_Code", "CAST(LATITUDE AS FLOAT) as Latitude",
    "CAST(LONGITUDE AS FLOAT) as Longitude"
)

df_chicago_sel = df_chicago.selectExpr(
    "ADDRESS as Address", "CITY", "STREET_DIRECTION",
    "CAST(ZIP AS STRING) as Zip_Code",
    "CAST(LATITUDE AS FLOAT) as Latitude",
    "CAST(LONGITUDE AS FLOAT) as Longitude"
)



In [0]:
from pyspark.sql.functions import col, trim, upper, round

# Combine data
df_combined = df_dallas_sel.unionByName(df_chicago_sel)

# Normalize text and round numeric columns
df_cleaned = df_combined.filter(
    col("Address").isNotNull() &
    col("City").isNotNull() &
    col("Latitude").isNotNull() &
    col("Longitude").isNotNull()
).withColumn("Address", trim(upper(col("Address")))) \
 .withColumn("City", trim(upper(col("City")))) \
 .withColumn("Street_Direction", trim(upper(col("Street_Direction")))) \
 .withColumn("Zip_Code", trim(col("Zip_Code"))) \
 .withColumn("Latitude", round(col("Latitude"), 5)) \
 .withColumn("Longitude", round(col("Longitude"), 5))

# Remove duplicates based on all cleaned fields
df_dedup = df_cleaned.dropDuplicates([
    "Address", "City", "Street_Direction", "Zip_Code", "Latitude", "Longitude"
])


In [0]:
from pyspark.sql.functions import row_number, current_date, lit
from pyspark.sql.window import Window

windowSpec = Window.orderBy("Address", "City", "Street_Direction", "Zip_Code", "Latitude", "Longitude")

df_final = df_dedup.withColumn("Location_SK", row_number().over(windowSpec)) \
                   .withColumn("DI_Job_ID", lit(1001)) \
                   .withColumn("DI_Load_Date", current_date())


In [0]:
(
    df_final.select(
        "Location_SK", "Address", "City", "Street_Direction",
        "Zip_Code", "Latitude", "Longitude", "DI_Job_ID", "DI_Load_Date"
    )
    .write
    .format("snowflake")
    .options(**snowflake_options)
    .option("dbtable", "DIM_LOCATION")
    .mode("overwrite")  # use "append" if needed
    .save()
)


In [0]:
df_check = spark.read \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "DIM_LOCATION") \
    .load()

df_check.orderBy("Location_SK").show(50, truncate=False)


+-----------+---------------------+-------+----------------+--------+--------+---------+---------+------------+
|LOCATION_SK|ADDRESS              |CITY   |STREET_DIRECTION|ZIP_CODE|LATITUDE|LONGITUDE|DI_JOB_ID|DI_LOAD_DATE|
+-----------+---------------------+-------+----------------+--------+--------+---------+---------+------------+
|1          |0 FAIR PARK          |DALLAS |UNKNOWN         |75226   |0       |0        |1001     |2025-04-19  |
|2          |1 E 113TH ST         |CHICAGO|EAST            |60628.0 |41.68885|-87.62289|1001     |2025-04-19  |
|3          |1 E 83RD ST          |CHICAGO|EAST            |60619.0 |41.74355|-87.62425|1001     |2025-04-19  |
|4          |1 E DELAWARE PL      |CHICAGO|EAST            |60611.0 |41.89903|-87.62819|1001     |2025-04-19  |
|5          |1 E ERIE ST          |CHICAGO|EAST            |60611.0 |41.89396|-87.62806|1001     |2025-04-19  |
|6          |1 E JACKSON BLVD     |CHICAGO|EAST            |60604.0 |41.87811|-87.62753|1001     |2025-0